# Build, execute and train a circuit

Construct a three-wire probability model from a feature map, an ansatz and a measurement. First call the compiled QNode with explicit inputs and weights. Then give PyTorch ownership of the same weights and optimize an outcome probability.

Run the cells in order with QASP 0.1.2. See the [tutorial setup guide](../README.md) for installation and optional PDF rendering.

In [ ]:
%matplotlib inline
from pathlib import Path
from importlib.metadata import version
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

print("QASP", version("qasp"))
np.random.seed(17)
torch.manual_seed(17)

## Encode three features

Each Pauli word consumes one input. The first feature rotates wire 0 about Y; the next two drive YY rotations across the specified pairs. Enabling input scaling makes each encoded angle a scale times a feature.

In [ ]:
import numpy as np
from qasp import AngleEmbedding, AngleAnsatz, CircuitBuilder

n = 3
embedding = AngleEmbedding(
    n, features=3, rotations=["Y 0", "YY 0 1", "YY 1 2"], input_scaling=True,
)
x = np.array([0.2, -0.1, 0.4], dtype=np.float64)

## Add trainable rotations and measurements

Two variational layers each apply Z and Y on three wires, giving twelve variational angles. Reuploading assigns three input scales to each encoding, for six scales in total. CNOTs on the supplied edges entangle the wires. Measuring all three returns eight basis probabilities.

In [ ]:
ansatz = AngleAnsatz(
    n, n_layers=2, feature_map=embedding, rotations="ZY",
    include_imprimitive=True, imprimitive="CNOT",
    imprimitive_pattern=[(0, 1), (0, 2), (1, 2)], reuploading=True,
)

measurement_type = "probs"
measurement_options = {"wires": [0, 1, 2]}
builder = CircuitBuilder(n)
builder.set_ansatz(ansatz)
builder.set_measurement(measurement_type, **measurement_options)
builder.build()

## Call the QNode directly

The returned function accepts one data vector plus both parameter groups at every call. Supply arrays in the order defined by the template. Print the shapes when changing the number of rotations or layers.

In [ ]:
print("Input-scale and angle shapes:", ansatz.get_param_shapes())
node = builder.compile(device="auto")

input_scaling_params = np.ones(6, dtype=np.float64)
params = np.array([
    0.1, 0.2, 0.3, 0.4, 0.5, 0.6,
    0.7, 0.8, 0.9, 1.0, 1.2, 1.4,
], dtype=np.float64)
probabilities = node(x, input_scaling_params=input_scaling_params, params=params)
print(probabilities)
assert probabilities.shape == (8,)
np.testing.assert_allclose(probabilities.sum(), 1.0)

## Initialize a TorchLayer

Pass concrete weights to reproduce the direct call. Alternatively, supply the two initializer objects below to start a new model. The low-level builder spells the angle keyword `initialization`; the model factories use `init`.

In [ ]:
from qasp import Normal, Uniform

x_torch = torch.as_tensor(x)
matched = builder.create_torchlayer(params=params, input_scaling_params=input_scaling_params).double()
np.testing.assert_allclose(matched(x_torch).detach(), probabilities, atol=1e-7)

weights = {"initialization": Normal(0, 0.2), "input_scaling_init": Uniform(0.8, 1.2)}
layer = builder.create_torchlayer(**weights).double()
print("Angles:", layer.params.detach())
print("Input scales:", layer.input_scaling_params.detach())

## Optimize and inspect the result

Train the layer to increase the probability of 000 on the fixed input. Adam receives both weight groups through layer.parameters(). Read the probability before and after the update rather than interpreting the loss as a probability.

In [ ]:
optimizer = torch.optim.Adam(layer.parameters(), lr=0.04)
before = layer(x_torch).detach()
losses = []
for step in range(12):
    optimizer.zero_grad()
    loss = -layer(x_torch)[0].clamp_min(1e-12).log()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
after = layer(x_torch).detach()
print("P(000), before and after:", before[0].item(), after[0].item())
assert torch.isfinite(after).all() and after[0] > before[0]
plt.plot(range(1, len(losses)+1), losses, "o-")
plt.xlabel("Adam step")
plt.ylabel("Negative log probability")
plt.show()

## Inspect the circuit

Use the trained weights and the same representative input for both renderers. Matplotlib is the default for the module drawing helpers.

In [ ]:
import pennylane as qml

args = {"params": layer.params, "input_scaling_params": layer.input_scaling_params}
fig, ax = qml.draw_mpl(node, decimals=2)(x_torch, **args)
display(fig)
plt.close(fig)

### Preview the paper figure

Use the same data and trained weights for a standalone vector PDF. Numeric labels on the explicit Y and YY encoding gates show the scaled feature angles; the Z and Y ansatz gates show their fitted angles. Set decimals to control the printed precision.

The helper uses temporary build files. To keep the displayed source, explicitly run `Path("circuit.tex").write_text(tex, encoding="utf-8")`, or call `builder.save_tikz(...)` when working with a builder.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import shutil
import subprocess
from IPython.display import Image, display


def show_tikz(source, name="Quantum circuit"):
    """Preview standalone TikZ inline. The optional name is the image description."""
    gs = shutil.which("gs") or shutil.which("gswin64c")
    if not shutil.which("pdflatex") or not gs:
        print("TikZ preview requires pdfLaTeX with Quantikz and Ghostscript.")
        return

    with TemporaryDirectory() as tmp:
        (Path(tmp) / "circuit.tex").write_text(source, encoding="utf-8")
        latex = subprocess.run(
            ["pdflatex", "-no-shell-escape", "-interaction=nonstopmode",
             "-halt-on-error", "circuit.tex"], cwd=tmp, capture_output=True, text=True,
        )
        if latex.returncode:
            raise RuntimeError(latex.stdout[-2000:])
        png = subprocess.check_output(
            [gs, "-q", "-dSAFER", "-sDEVICE=png16m", "-dTextAlphaBits=4", "-dGraphicsAlphaBits=4", "-r300", "-o", "-", "circuit.pdf"],
            cwd=tmp,
        )
    # Keep the original display width. Specifying only width preserves the aspect ratio.
    width_px = int.from_bytes(png[16:20], "big")  # PNG IHDR width
    display(Image(data=png, width=max(1, round(width_px * 120 / 300)), alt=name))

In [ ]:
tex = builder.to_tikz(x_torch, **args, standalone=True, symbolic_parameters=False, decimals=2)
show_tikz(tex, "building_blocks")

## Try it

Switch the readout to a marginal on wires 0 and 2. Rebuild and recompile the circuit. How many outputs does the QNode return, and what event does output zero now represent?